# 01 — Tải cấu trúc và metadata từ Materials Project
Lưu `material_id`, `task_id`, `run_type`, phiên bản database và thiết lập tính toán của task tạo cấu trúc.

**Đầu ra:** `data/raw/mp_manifest.csv`, `data/raw/cif/<material_id>.cif`, `data/raw/mp_settings.jsonl` (settings đầy đủ).

In [ ]:
# Cài thư viện (chỉ chạy trên Colab; runtime Colab bị reset sau mỗi phiên nên mỗi notebook tự cài thứ nó cần)
import sys
if "google.colab" in sys.modules:
    %pip install -q mp-api pymatgen

In [ ]:
import os, sys, json, time
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/ml_xc_colab"
else:
    ROOT = os.environ.get("ML_XC_ROOT", os.getcwd())
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)          # để import được thư mục utils/
DATA, RAW, PROC = f"{ROOT}/data", f"{ROOT}/data/raw", f"{ROOT}/data/processed"
print("ROOT =", ROOT)

## Tham số truy vấn
`STABLE_ONLY = True` chỉ lấy vật liệu nằm trên convex hull, nên `energy_above_hull` của toàn bộ mẫu bằng 0 (cột này sẽ không mang thông tin). Đặt `STABLE_ONLY = False` và `E_HULL_MAX` (eV/nguyên tử) nếu muốn có vật liệu cận ổn định.

In [ ]:
N_MATERIALS = 1000
NUM_SITES = (1, 20)        # số nguyên tử trong ô đơn vị; bước 02 sẽ lọc chặt hơn theo chi phí DFT
STABLE_ONLY = True
E_HULL_MAX = None          # ví dụ 0.05 khi STABLE_ONLY = False

In [ ]:
from utils import mp_api_utils as mpu

api_key = mpu.get_api_key("MP_API_KEY")      # Colab Secrets -> biến môi trường -> nhập ẩn
rows, db_version = mpu.download_mp(api_key, n_materials=N_MATERIALS, num_sites=NUM_SITES,
                                   stable_only=STABLE_ONLY, e_hull_max=E_HULL_MAX)
del api_key
print("Phiên bản database:", db_version, "| số vật liệu:", len(rows))

In [ ]:
manifest = mpu.save_raw(rows, RAW)
print(f"Đã lưu {len(manifest)} dòng vào {RAW}/mp_manifest.csv và {len(manifest)} file CIF")
print("\nrun_type:")
print(manifest["run_type"].value_counts(dropna=False))
print("\ntask_type:")
print(manifest["task_type"].value_counts(dropna=False))
print("\nenergy_above_hull: min/max =", manifest["energy_above_hull_eV_per_atom"].min(),
      manifest["energy_above_hull_eV_per_atom"].max())
manifest.head()